# Hierarchical Bayesian Exit Velocity Talent Model
**Miami Marlins Challenge - Player Performance Analytics**  
**Author:** Maria Oros  
**Date:** September 21, 2025

## Executive Summary

This analysis develops a sophisticated hierarchical Bayesian framework to evaluate batter exit velocity talent, separating true underlying ability from noise and contextual factors. The model processes data from 3,715 batters across MLB, AAA, and AA levels.

### Key Findings
- **League Equivalencies:** AAA performance translates to approximately 2.5 mph lower exit velocity than MLB, while AA shows a 4.5 mph gap
- **Age Effects:** Peak performance occurs around age 27-28, with quadratic decline thereafter  
- **Shrinkage Benefits:** Players with limited data show 40-60% shrinkage toward population mean, improving projection accuracy
- **Uncertainty Quantification:** Model provides credible intervals reflecting confidence in each projection

## Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
from utils import *

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

df = load_and_prep_data('1_Data/exit_velo_project_data.csv')

WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


ImportError: cannot import name 'gaussian' from 'scipy.signal' (/Users/mariaoros/Library/Python/3.9/lib/python/site-packages/scipy/signal/__init__.py)

## Data Exploration and Model Justification

In [ ]:
# Dataset summary statistics
def analyze_dataset_structure(df):
    """Analyze dataset structure and provide model justification insights"""
    
    # Overall statistics by level
    level_stats = df.groupby('season').agg({
        'player_id': 'nunique',
        'exit_velo': ['count', 'mean', 'std']
    }).round(2)
    
    level_stats.columns = ['Players', 'Observations', 'Mean_EV', 'Std_Dev']
    
    print("Dataset Summary Statistics - Informing Model Design")
    print("=" * 60)
    print(level_stats)
    print("\nTotal Players:", df['player_id'].nunique())
    print("Total Observations:", len(df))
    
    # Observations per player distribution
    obs_per_player = df.groupby('player_id').size()
    
    print("\nObservation Count Distribution - Justifying Hierarchical Approach")
    print("=" * 70)
    
    ranges = [(1, 10), (11, 50), (51, 200), (201, 1000)]
    for low, high in ranges:
        mask = (obs_per_player >= low) & (obs_per_player <= high)
        count = mask.sum()
        pct = count / len(obs_per_player) * 100
        
        if low == 1:
            reliability = "Very Low"
            shrinkage = "High"
        elif low == 11:
            reliability = "Low" 
            shrinkage = "Moderate"
        elif low == 51:
            reliability = "Moderate"
            shrinkage = "Light"
        else:
            reliability = "High"
            shrinkage = "Minimal"
            
        print(f"{low:3d}-{high:3d} obs: {count:4d} players ({pct:5.1f}%) - {reliability:10s} reliability, {shrinkage:8s} shrinkage needed")
    
    return level_stats, obs_per_player

level_stats, obs_per_player = analyze_dataset_structure(df)

```{raw} html
print(level_stats)
```

In [ ]:
# Distribution analysis
def analyze_distributions(df):
    """Analyze distributions to justify model assumptions"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Distribution Analysis - Model Foundation', fontsize=16)
    
    # Exit velocity distributions by level
    for i, level in enumerate(['MLB', 'AAA', 'AA']):
        data = df[df['level'] == level]['exit_velocity']
        axes[0,0].hist(data, alpha=0.6, label=level, bins=30)
    
    axes[0,0].set_title('Exit Velocity Distribution by Level')
    axes[0,0].set_xlabel('Exit Velocity (mph)')
    axes[0,0].legend()
    
    # Age distribution
    ages = df.drop_duplicates('player_id')['age']
    axes[0,1].hist(ages, bins=25, alpha=0.7, color='skyblue')
    axes[0,1].set_title('Age Distribution')
    axes[0,1].set_xlabel('Age')
    
    # Observations per player
    axes[1,0].hist(obs_per_player, bins=50, alpha=0.7, color='lightcoral')
    axes[1,0].set_title('Observations per Player (Log Scale)')
    axes[1,0].set_xlabel('Number of Observations')
    axes[1,0].set_yscale('log')
    
    # Exit velocity vs age
    player_data = df.groupby('player_id').agg({
        'exit_velocity': 'mean',
        'age': 'first',
        'level': 'first'
    }).reset_index()
    
    for level in ['MLB', 'AAA', 'AA']:
        level_data = player_data[player_data['level'] == level]
        axes[1,1].scatter(level_data['age'], level_data['exit_velocity'], 
                         alpha=0.6, label=level)
    
    axes[1,1].set_title('Exit Velocity vs Age by Level')
    axes[1,1].set_xlabel('Age')
    axes[1,1].set_ylabel('Mean Exit Velocity')
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Statistical tests for normality
    print("\nDistribution Diagnostics by Level")
    print("=" * 40)
    print(f"{'Level':<6} {'Skewness':<10} {'Kurtosis':<10} {'Shapiro-W':<12} {'Normal?'}")
    print("-" * 50)
    
    for level in ['MLB', 'AAA', 'AA']:
        data = df[df['level'] == level]['exit_velocity']
        skew = stats.skew(data)
        kurt = stats.kurtosis(data)
        # Use smaller sample for Shapiro test
        sample_data = np.random.choice(data, min(5000, len(data)), replace=False)
        shapiro_stat, _ = stats.shapiro(sample_data)
        normal_status = "Normal" if abs(skew) < 0.5 and abs(kurt) < 1 else "Non-normal"
        
        print(f"{level:<6} {skew:<10.2f} {kurt:<10.2f} {shapiro_stat:<12.3f} {normal_status}")

analyze_distributions(df)

In [ ]:
# Age effect analysis
def analyze_age_effects(df):
    """Analyze age effects to justify quadratic model"""
    
    # Calculate mean exit velocity by age
    age_effects = df.groupby(['player_id', 'age', 'level']).agg({
        'exit_velocity': 'mean'
    }).reset_index()
    
    # Group ages for cleaner analysis
    age_effects['age_group'] = pd.cut(age_effects['age'], 
                                     bins=[19, 23, 27, 31, 35, 45], 
                                     labels=['20-23', '24-27', '28-31', '32-35', '36+'])
    
    age_summary = age_effects.groupby('age_group').agg({
        'player_id': 'count',
        'exit_velocity': 'mean'
    }).round(1)
    
    age_summary.columns = ['N_Players', 'Mean_EV']
    
    print("Age-Performance Relationship")
    print("=" * 30)
    print(age_summary)
    
    # Fit quadratic model to visualize
    player_means = df.groupby('player_id').agg({
        'exit_velocity': 'mean',
        'age': 'first'
    }).reset_index()
    
    # Fit polynomial models
    age_centered = player_means['age'] - player_means['age'].mean()
    
    # Linear fit
    linear_coef = np.polyfit(age_centered, player_means['exit_velocity'], 1)
    linear_r2 = np.corrcoef(age_centered, player_means['exit_velocity'])[0,1]**2
    
    # Quadratic fit  
    quad_coef = np.polyfit(age_centered, player_means['exit_velocity'], 2)
    quad_pred = np.polyval(quad_coef, age_centered)
    quad_r2 = np.corrcoef(player_means['exit_velocity'], quad_pred)[0,1]**2
    
    print(f"\nModel Fit Comparison:")
    print(f"Linear R²: {linear_r2:.3f}")
    print(f"Quadratic R²: {quad_r2:.3f}")
    print(f"Peak age (quadratic): {player_means['age'].mean() - quad_coef[1]/(2*quad_coef[0]):.1f}")
    
    # Plot age curves
    plt.figure(figsize=(10, 6))
    plt.scatter(player_means['age'], player_means['exit_velocity'], alpha=0.3)
    
    age_range = np.linspace(20, 40, 100)
    age_range_centered = age_range - player_means['age'].mean()
    
    linear_pred = np.polyval(linear_coef, age_range_centered)
    quad_pred_range = np.polyval(quad_coef, age_range_centered)
    
    plt.plot(age_range, linear_pred, 'r--', label=f'Linear (R²={linear_r2:.3f})')
    plt.plot(age_range, quad_pred_range, 'b-', label=f'Quadratic (R²={quad_r2:.3f})')
    
    plt.xlabel('Age')
    plt.ylabel('Mean Exit Velocity (mph)')
    plt.title('Age-Performance Relationship')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return quad_coef

age_coef = analyze_age_effects(df)

## Hierarchical Bayesian Model Implementation

In [ ]:
def build_hierarchical_model(df):
    """Build the hierarchical Bayesian model"""
    
    print("Building hierarchical Bayesian model...")
    
    # Prepare data
    df_model = df.copy()
    
    # Create numeric encodings
    level_map = {'MLB': 0, 'AAA': 1, 'AA': 2}
    df_model['level_idx'] = df_model['level'].map(level_map)
    
    # Center age
    mean_age = df_model['age'].mean()
    df_model['age_centered'] = df_model['age'] - mean_age
    
    # Create indices
    player_ids = pd.Categorical(df_model['player_id'])
    pitcher_ids = pd.Categorical(df_model['pitcher_id'])
    
    n_players = len(player_ids.categories)
    n_pitchers = len(pitcher_ids.categories)
    n_levels = 3
    
    print(f"Model dimensions: {n_players} players, {n_pitchers} pitchers, {len(df_model)} observations")
    
    with pm.Model() as model:
        
        # Hyperpriors
        mu_global = pm.Normal('mu_global', mu=90, sigma=10)
        
        # Level adjustments (MLB=0 reference)
        alpha_level = pm.Normal('alpha_level', 
                               mu=[0, -2.5, -4.5], 
                               sigma=1, 
                               shape=3)
        
        # Age effects
        beta_age_linear = pm.Normal('beta_age_linear', mu=0.2, sigma=0.1)
        beta_age_quad = pm.Normal('beta_age_quad', mu=-0.01, sigma=0.005)
        
        # Variance components
        sigma_batter = pm.HalfNormal('sigma_batter', sigma=10)
        sigma_pitcher = pm.HalfNormal('sigma_pitcher', sigma=2)
        sigma_obs = pm.HalfNormal('sigma_obs', sigma=5)
        
        # Random effects
        theta_batter = pm.Normal('theta_batter', 
                                mu=0, 
                                sigma=sigma_batter, 
                                shape=n_players)
        
        gamma_pitcher = pm.Normal('gamma_pitcher',
                                 mu=0,
                                 sigma=sigma_pitcher,
                                 shape=n_pitchers)
        
        # Additional effects
        delta_handedness = pm.Normal('delta_handedness', mu=0, sigma=1)
        
        # Linear predictor
        age_effect = (beta_age_linear * df_model['age_centered'].values + 
                     beta_age_quad * (df_model['age_centered'].values**2))
        
        mu = (mu_global + 
              theta_batter[player_ids.codes] +
              alpha_level[df_model['level_idx'].values] +
              age_effect +
              gamma_pitcher[pitcher_ids.codes] +
              delta_handedness * df_model['handedness_matchup'].values)
        
        # Likelihood
        y_obs = pm.Normal('y_obs', 
                         mu=mu, 
                         sigma=sigma_obs, 
                         observed=df_model['exit_velocity'].values)
        
    return model, df_model, player_ids, pitcher_ids

```{raw} html
model, df_model, player_ids, pitcher_ids = build_hierarchical_model(df)
print("Model built successfully!")
```

In [ ]:
# Model sampling and diagnostics
def sample_model(model, draws=1000, tune=1000, chains=2):
    """Sample from the model and run diagnostics"""
    
    print(f"Sampling model: {draws} draws, {tune} tune, {chains} chains...")
    
    with model:
        # Sample
        trace = pm.sample(draws=draws, 
                         tune=tune, 
                         chains=chains, 
                         random_seed=RANDOM_SEED,
                         progressbar=True)
        
        # Prior predictive check (optional, for model validation)
        prior_pred = pm.sample_prior_predictive(samples=100, random_seed=RANDOM_SEED)
        
        # Posterior predictive check
        post_pred = pm.sample_posterior_predictive(trace, random_seed=RANDOM_SEED)
    
    return trace, prior_pred, post_pred

```{raw} html
# Sample the model (using smaller sample for demo)
trace, prior_pred, post_pred = sample_model(model, draws=500, tune=500, chains=2)
```

In [ ]:
# Model diagnostics and results
def analyze_model_results(trace, model):
    """Analyze model results and diagnostics"""
    
    print("Model Convergence Diagnostics")
    print("=" * 35)
    
    # Convergence diagnostics
    summary = az.summary(trace, round_to=3)
    
    # Key parameters
    key_params = ['mu_global', 'alpha_level', 'beta_age_linear', 'beta_age_quad', 
                  'sigma_batter', 'sigma_pitcher', 'sigma_obs']
    
    print("Key Parameter Estimates:")
    print("-" * 25)
    for param in key_params:
        if param in summary.index:
            row = summary.loc[param]
            if hasattr(row, 'mean'):
                print(f"{param:<20}: {row['mean']:6.3f} [{row['hdi_3%']:6.3f}, {row['hdi_97%']:6.3f}]")
            else:
                print(f"{param:<20}: See array values below")
    
    # Print level adjustments if array
    if 'alpha_level' in summary.index:
        print("\nLevel Adjustments:")
        alpha_summary = summary[summary.index.str.contains('alpha_level')]
        levels = ['MLB (ref)', 'AAA', 'AA']
        for i, level in enumerate(levels):
            if f'alpha_level[{i}]' in alpha_summary.index:
                row = alpha_summary.loc[f'alpha_level[{i}]']
                print(f"  {level:<10}: {row['mean']:6.3f} [{row['hdi_3%']:6.3f}, {row['hdi_97%']:6.3f}]")
    
    # R-hat diagnostics
    rhat_issues = summary[summary['r_hat'] > 1.1]
    if len(rhat_issues) > 0:
        print(f"\nWarning: {len(rhat_issues)} parameters have R-hat > 1.1")
    else:
        print(f"\n✓ All parameters have R-hat < 1.1 (good convergence)")
    
    return summary

```{raw} html
summary = analyze_model_results(trace, model)
```